# Certificación Profesional en Desarrollador de Agentes de IA
## Clase 3 — Salidas estructuradas y aplicaciones robustas

ITBA Educación Ejecutiva

Generar resultados que otros sistemas puedan consumir de forma confiable — no texto libre que hay que parsear a mano y cruzar los dedos.

### Cómo se usa este notebook

- **Durante la clase en vivo**: el docente corre y explica cada celda (demo con AWS Bedrock). Los alumnos no programan en simultáneo.
- **Celdas 🏠 Practicá**: tarea individual para después de la clase.
- **Sección 9 (Laboratorio)**: tarea principal de la semana — checkpoint del proyecto integrador. Se resuelve con el proveedor de cada uno (Bedrock, Anthropic, OpenAI, Google u Ollama), no hace falta que sea Bedrock.

## Objetivo de la clase

Al final de la clase vas a poder:
- Definir un JSON Schema y un modelo Pydantic equivalente para validar salidas de un LLM.
- Obtener salidas estructuradas usando el mecanismo nativo del proveedor cuando existe, y un patrón de reintento universal cuando no.
- Usar tool calling para que el modelo pida ejecutar una función externa con argumentos validados.
- Configurar timeouts, reintentos con backoff+jitter, y registrar tokens/costo/latencia de cada llamada.
- Consumir una respuesta en streaming.
- Refactorizar el cliente de la Clase 2 detrás de una interfaz común (`LLMProvider`), agnóstica de proveedor.

**Checkpoint del proyecto integrador de hoy**: servicio que devuelve salidas estructuradas y validadas (Pydantic) a partir de texto libre, sobre el cliente ya agnóstico de proveedor (ver `proyecto-integrador/README.md`, checkpoint Clase 3).

## Agenda

1. JSON Schema y Pydantic
2. Structured outputs (mecanismo nativo del proveedor)
3. Manejo de respuestas inválidas (patrón universal de reintento)
4. Tool calling
5. Timeouts, reintentos y backoff
6. Observabilidad básica: tokens, costo y latencia
7. Streaming
8. Abstracción de proveedores
9. Laboratorio: servicio de extracción estructurada

## 0. Punto de partida

Recordá las funciones `llamar_bedrock`, `llamar_anthropic`, `llamar_openai`, `llamar_google` y `llamar_ollama` de la Clase 2 — hoy las vamos a envolver en clases con una interfaz común. Para que este notebook sea autocontenido, las volvemos a definir acá (son las mismas que ya tenías).

In [ ]:
import os
import json
import time
import random
from dotenv import load_dotenv

load_dotenv()
provider = os.environ.get("LLM_PROVIDER", "bedrock")
print(f"Proveedor configurado: {provider}")

In [ ]:
# --- Funciones de la Clase 2 (recordatorio) ---
import boto3

def llamar_bedrock(prompt, system=None, temperature=0.7, max_tokens=500):
    client = boto3.client("bedrock-runtime", region_name=os.environ.get("AWS_REGION", "us-east-1"))
    kwargs = {"modelId": os.environ.get("BEDROCK_MODEL_ID_CLAUDE", "global.anthropic.claude-sonnet-4-6"),
              "messages": [{"role": "user", "content": [{"text": prompt}]}],
              "inferenceConfig": {"temperature": temperature, "maxTokens": max_tokens}}
    if system:
        kwargs["system"] = [{"text": system}]
    resp = client.converse(**kwargs)
    return resp["output"]["message"]["content"][0]["text"]

bedrock = boto3.client("bedrock-runtime", region_name=os.environ.get("AWS_REGION", "us-east-1"))
model_id = os.environ.get("BEDROCK_MODEL_ID_CLAUDE", "global.anthropic.claude-sonnet-4-6")

## 1. JSON Schema y Pydantic

- **JSON Schema**: un documento (también JSON) que describe la forma esperada de otro documento JSON — tipos, campos requeridos, enums, etc. Es el estándar que entienden las APIs de los proveedores para "forzar" un formato de salida.
- **Pydantic**: librería de Python que define esa misma forma como una clase, valida datos contra ella, y puede generar el JSON Schema automáticamente — no hace falta escribirlo dos veces.

In [ ]:
# %pip install pydantic  # si hace falta instalarlo

from pydantic import BaseModel, Field, ValidationError
from typing import Literal

class ConsultaSoporte(BaseModel):
    categoria: Literal["facturacion", "tecnico", "cuenta", "otro"]
    urgencia: Literal["baja", "media", "alta"]
    resumen: str = Field(..., max_length=200, description="Resumen breve del reclamo, en una oración")


# El JSON Schema se genera solo a partir de la clase:
schema = ConsultaSoporte.model_json_schema()
print(json.dumps(schema, indent=2, ensure_ascii=False))

In [ ]:
# Validación: un dict válido pasa, uno inválido lanza ValidationError con el detalle exacto de qué falló
dato_valido = {"categoria": "tecnico", "urgencia": "alta", "resumen": "El servicio no responde desde hace 2 horas."}
consulta = ConsultaSoporte(**dato_valido)
print(consulta)

try:
    ConsultaSoporte(categoria="inventado", urgencia="alta", resumen="x")
except ValidationError as e:
    print("\nError de validación (esperado):")
    print(e)

**🏠 Practicá (tarea, después de la clase)**: agregá un campo `requiere_escalamiento: bool` al modelo `ConsultaSoporte`. Regenerá el JSON Schema y confirmá que el nuevo campo aparece en `properties`. Probá instanciar el modelo sin ese campo — ¿qué pasa? (pista: Pydantic trata los campos sin default como requeridos).

## 2. Structured outputs

Pedirle al modelo "devolveme JSON" en el prompt **no garantiza** que la salida sea JSON válido, ni que cumpla el schema — el modelo puede agregar texto antes/después, usar comillas simples, olvidar un campo, etc.

AWS Bedrock (igual que OpenAI y algunos otros proveedores) tiene un mecanismo **nativo** que fuerza al modelo a producir una salida que cumple exactamente el schema: en Bedrock, el campo `outputConfig.textFormat` de la Converse API.

In [ ]:
schema_bedrock = ConsultaSoporte.model_json_schema()
schema_bedrock["additionalProperties"] = False  # requerido por el validador de Bedrock para objetos "cerrados"

texto_libre = "Hace dos días que no puedo entrar a mi cuenta y ya perdí una reunión importante por esto, necesito que lo resuelvan YA."

response = bedrock.converse(
    modelId=model_id,
    messages=[{"role": "user", "content": [{"text": f"Clasificá esta consulta de soporte: '{texto_libre}'"}]}],
    outputConfig={
        "textFormat": {
            "type": "json_schema",
            "structure": {
                "jsonSchema": {
                    "name": "consulta_soporte",
                    "description": "Clasificación estructurada de una consulta de soporte",
                    "schema": json.dumps(schema_bedrock),
                }
            }
        }
    },
)

raw_json = response["output"]["message"]["content"][0]["text"]
print("JSON crudo devuelto por el modelo:", raw_json)

consulta_validada = ConsultaSoporte.model_validate_json(raw_json)
print("\nObjeto Python validado:", consulta_validada)

Notá que igual pasamos el JSON por `ConsultaSoporte.model_validate_json` aunque Bedrock ya garantiza el schema — no es redundante: valida también las reglas que un JSON Schema básico no expresa bien (en este caso ninguna, pero en general sí: por ejemplo relaciones entre campos, o validadores custom de Pydantic).

**🏠 Practicá**: cambiá `texto_libre` por una consulta de facturación de baja urgencia y confirmá que `categoria` y `urgencia` salen correctas sin que se lo digas explícitamente — el modelo infiere la clasificación, no la copia del texto.

## 3. Manejo de respuestas inválidas

No todos los proveedores tienen un mecanismo nativo de structured outputs (Anthropic directo, Google y Ollama no lo tienen igual de simple). Para esos casos, o como red de seguridad general, conviene un patrón universal: **pedir JSON en el prompt, parsear, validar con Pydantic, y si falla, reintentar mostrándole al modelo el error exacto** para que se corrija.

In [ ]:
def generar_estructurado_con_reintentos(llamar_fn, prompt, schema_model, system=None, max_reintentos=3):
    """Patrón universal: sirve con cualquier función llamar_* (Bedrock, OpenAI, Anthropic, Google, Ollama)."""
    instruccion_base = (
        f"{prompt}\n\nRespondé ÚNICAMENTE con un JSON válido que cumpla este schema "
        f"(sin texto adicional, sin markdown):\n{schema_model.model_json_schema()}"
    )
    ultimo_error = None

    for intento in range(max_reintentos):
        instruccion = instruccion_base
        if ultimo_error:
            instruccion += f"\n\nTu respuesta anterior no cumplió el schema. Error: {ultimo_error}\nCorregí y devolvé SOLO el JSON."

        texto = llamar_fn(instruccion, system=system, temperature=0.0)  # temperature baja: queremos formato consistente, no creatividad
        try:
            return schema_model.model_validate_json(texto)
        except (ValidationError, json.JSONDecodeError) as e:
            ultimo_error = str(e)
            print(f"Intento {intento + 1} inválido, reintentando...")

    raise ValueError(f"No se obtuvo una respuesta válida tras {max_reintentos} intentos. Último error: {ultimo_error}")


# Prueba con la función de Bedrock, sin usar el mecanismo nativo (simula un proveedor sin structured outputs)
resultado = generar_estructurado_con_reintentos(
    llamar_bedrock,
    f"Clasificá esta consulta de soporte: '{texto_libre}'",
    ConsultaSoporte,
)
print(resultado)

**🏠 Practicá**: bajá `max_reintentos` a 1 y modificá el prompt para pedir un campo que no existe en el schema (ej. "incluí también un campo 'telefono'") — el modelo probablemente va a devolver un campo extra o mal formado. Observá el error de Pydantic y confirmá que la función lo reporta con `raise ValueError`, no con un traceback críptico.

## 4. Tool calling

Tool calling es un caso particular de salida estructurada: en vez de pedirle al modelo un objeto de negocio, le das una lista de **herramientas disponibles** (funciones con su schema de argumentos) y el modelo decide *si* necesita alguna y con qué argumentos — vos ejecutás la función real y le devolvés el resultado.

El modelo **nunca ejecuta código** — solo pide "llamá a esta función con estos argumentos". La ejecución es responsabilidad de tu aplicación (esto es clave para seguridad, lo retomamos en la Clase 9 y la Clase 11).

In [ ]:
# La "herramienta" real: una función Python común y corriente
def consultar_estado_ticket(ticket_id: str) -> dict:
    tickets_mock = {"T-001": "En progreso", "T-002": "Resuelto", "T-003": "Esperando al cliente"}
    return {"ticket_id": ticket_id, "estado": tickets_mock.get(ticket_id, "No encontrado")}


# La descripción de esa función que le mandamos al modelo (JSON Schema de sus argumentos)
tool_config = {
    "tools": [{
        "toolSpec": {
            "name": "consultar_estado_ticket",
            "description": "Consulta el estado actual de un ticket de soporte por su ID.",
            "inputSchema": {"json": {
                "type": "object",
                "properties": {
                    "ticket_id": {"type": "string", "description": "ID del ticket, ej. T-001"}
                },
                "required": ["ticket_id"],
            }},
        }
    }]
}

In [ ]:
messages = [{"role": "user", "content": [{"text": "¿Cuál es el estado del ticket T-002?"}]}]

response = bedrock.converse(modelId=model_id, messages=messages, toolConfig=tool_config)
messages.append(response["output"]["message"])

print("Stop reason:", response["stopReason"])

if response["stopReason"] == "tool_use":
    for block in response["output"]["message"]["content"]:
        if "toolUse" in block:
            tool = block["toolUse"]
            print(f"El modelo pide llamar a: {tool['name']}({tool['input']})")

            if tool["name"] == "consultar_estado_ticket":
                resultado = consultar_estado_ticket(**tool["input"])  # acá ejecutamos NOSOTROS la función real
                messages.append({
                    "role": "user",
                    "content": [{"toolResult": {"toolUseId": tool["toolUseId"], "content": [{"json": resultado}]}}],
                })

    # Segunda llamada: le devolvemos el resultado de la herramienta para que arme la respuesta final
    response_final = bedrock.converse(modelId=model_id, messages=messages, toolConfig=tool_config)
    print("\nRespuesta final:", response_final["output"]["message"]["content"][0]["text"])
else:
    print(response["output"]["message"]["content"][0]["text"])

**🏠 Practicá**: preguntá por un ticket que no existe (ej. "T-999") y confirmá que la respuesta final del modelo refleja el "No encontrado" que devuelve la función — no inventa un estado. Después agregá una segunda herramienta mock (ej. `escalar_ticket(ticket_id, motivo)`) y preguntá algo que dispare esa herramienta en vez de la primera.

## 5. Timeouts, reintentos y backoff

En la Clase 2 armamos `generar_con_reintentos` con backoff exponencial fijo (1s, 2s, 4s...). Dos mejoras para producción:

- **Timeout explícito**: sin él, una llamada colgada puede bloquear tu aplicación indefinidamente.
- **Jitter**: sumar un poco de aleatoriedad al backoff evita que muchos clientes reintenten exactamente al mismo tiempo tras un error masivo (efecto "estampida").

In [ ]:
from botocore.config import Config
from botocore.exceptions import ClientError

# Cliente con timeouts explícitos y sin el retry automático de boto3 (lo manejamos nosotros)
bedrock_prod = boto3.client(
    "bedrock-runtime",
    region_name=os.environ.get("AWS_REGION", "us-east-1"),
    config=Config(connect_timeout=5, read_timeout=15, retries={"max_attempts": 0}),
)


def generar_con_reintentos(bedrock_client, model_id, messages, system=None, inference_config=None, max_reintentos=3):
    kwargs = {"modelId": model_id, "messages": messages}
    if system:
        kwargs["system"] = system
    if inference_config:
        kwargs["inferenceConfig"] = inference_config

    for intento in range(max_reintentos):
        try:
            return bedrock_client.converse(**kwargs)
        except ClientError as err:
            codigo = err.response["Error"]["Code"]
            if codigo == "ThrottlingException" and intento < max_reintentos - 1:
                espera = (2 ** intento) + random.uniform(0, 1)  # backoff exponencial + jitter
                print(f"Rate limit, reintentando en {espera:.1f}s...")
                time.sleep(espera)
                continue
            if codigo == "ValidationException":
                raise ValueError(f"Request inválido: {err.response['Error']['Message']}") from err
            raise
    raise RuntimeError("Se agotaron los reintentos por throttling.")


resp = generar_con_reintentos(bedrock_prod, model_id, messages=[{"role": "user", "content": [{"text": "Decí 'ok'."}]}])
print(resp["output"]["message"]["content"][0]["text"])

**🏠 Practicá**: bajá `read_timeout` a 0.01 segundos y volvé a correr la celda anterior — debería fallar con un timeout de lectura. Envolvé la llamada en un `try/except` que capture ese error específico (`botocore.exceptions.ReadTimeoutError`) y decidí si tiene sentido reintentar (sí: un timeout es, en general, reintentable).

## 6. Observabilidad básica: tokens, costo y latencia

Sin esto, no hay forma de responder "¿cuánto nos está costando esto?" ni de detectar una llamada anormalmente lenta. Registrarlo desde el día 1 es mucho más barato que agregarlo después.

In [ ]:
# Valores de ejemplo — el precio real cambia con el tiempo, verificar en la consola de Bedrock antes de usar en producción
PRECIO_POR_MILLON_USD = {"input": 3.0, "output": 15.0}

def estimar_costo_usd(usage: dict) -> float:
    costo_input = usage["inputTokens"] / 1_000_000 * PRECIO_POR_MILLON_USD["input"]
    costo_output = usage["outputTokens"] / 1_000_000 * PRECIO_POR_MILLON_USD["output"]
    return costo_input + costo_output


registro_llamadas = []  # en una app real, esto va a un log estructurado o una base de datos (Clase 11)

def llamar_con_registro(bedrock_client, model_id, messages, **kwargs):
    resp = generar_con_reintentos(bedrock_client, model_id, messages, **kwargs)
    registro_llamadas.append({
        "input_tokens": resp["usage"]["inputTokens"],
        "output_tokens": resp["usage"]["outputTokens"],
        "costo_usd": estimar_costo_usd(resp["usage"]),
        "latencia_ms": resp["metrics"]["latencyMs"],
    })
    return resp

resp = llamar_con_registro(bedrock_prod, model_id, messages=[{"role": "user", "content": [{"text": "Explicá qué es un embedding en 1 oración."}]}])
print(resp["output"]["message"]["content"][0]["text"])
print("\nRegistro:", registro_llamadas[-1])

**🏠 Practicá**: hacé 5 llamadas distintas con `llamar_con_registro` y al final calculá el costo total y la latencia promedio a partir de `registro_llamadas` (con una simple suma/promedio sobre la lista de dicts). Esto es literalmente lo que vas a necesitar para el informe de costos del TP Final (Clase 12).

## 7. Streaming

Para respuestas largas, esperar a que termine toda la generación antes de mostrar nada da una sensación de latencia alta. El streaming muestra el texto a medida que se genera, token a token.

In [ ]:
stream_response = bedrock.converse_stream(
    modelId=model_id,
    messages=[{"role": "user", "content": [{"text": "Listá 5 buenas prácticas de seguridad para agentes de IA, con una oración de explicación cada una."}]}],
)

for event in stream_response["stream"]:
    if "contentBlockDelta" in event:
        print(event["contentBlockDelta"]["delta"]["text"], end="", flush=True)
    if "metadata" in event and "usage" in event["metadata"]:
        print(f"\n\n[tokens: {event['metadata']['usage']}]")

**🏠 Practicá**: medí con `time.time()` cuánto tarda en llegar el **primer** chunk de texto (time to first token) vs. cuánto tarda la respuesta completa con `converse` normal (sin streaming) para el mismo prompt. Esa diferencia es la que justifica usar streaming en una UI interactiva.

## 8. Abstracción de proveedores

Hasta acá todo el notebook (y el `LLMClient` de la Clase 2) llama directo a Bedrock, con un `if/elif` por proveedor en el mejor de los casos. Eso no escala: cada vez que agregás lógica nueva (reintentos, structured outputs, registro de costos) hay que replicarla en cada rama.

La solución: una **interfaz común** (clase abstracta) que cada proveedor implementa. El resto de la aplicación programa contra la interfaz, no contra un SDK específico.

In [ ]:
from abc import ABC, abstractmethod


class LLMProvider(ABC):
    """Interfaz común para cualquier proveedor de modelos de lenguaje."""

    @abstractmethod
    def generate(self, prompt: str, system: str = None, temperature: float = 0.7, max_tokens: int = 500) -> str:
        """Devuelve el texto generado por el modelo."""
        ...

    def generate_structured(self, prompt: str, schema_model, system: str = None):
        """Implementación por defecto: sirve para CUALQUIER proveedor, aunque no tenga
        soporte nativo de structured outputs. Un proveedor que sí lo tenga (como Bedrock)
        puede sobreescribir este método con una versión más eficiente (ver BedrockProvider)."""
        return generar_estructurado_con_reintentos(self.generate, prompt, schema_model, system=system)

In [ ]:
class BedrockProvider(LLMProvider):
    def __init__(self):
        self.client = boto3.client(
            "bedrock-runtime",
            region_name=os.environ.get("AWS_REGION", "us-east-1"),
            config=Config(connect_timeout=5, read_timeout=15, retries={"max_attempts": 0}),
        )
        self.model_id = os.environ.get("BEDROCK_MODEL_ID_CLAUDE", "global.anthropic.claude-sonnet-4-6")

    def generate(self, prompt, system=None, temperature=0.7, max_tokens=500):
        messages = [{"role": "user", "content": [{"text": prompt}]}]
        system_param = [{"text": system}] if system else None
        inference_config = {"temperature": temperature, "maxTokens": max_tokens}
        resp = generar_con_reintentos(self.client, self.model_id, messages, system=system_param, inference_config=inference_config)
        return resp["output"]["message"]["content"][0]["text"]

    def generate_structured(self, prompt, schema_model, system=None):
        # Override: usa el mecanismo nativo de Bedrock (sección 2) en vez del reintento genérico
        schema = schema_model.model_json_schema()
        schema["additionalProperties"] = False
        kwargs = {
            "modelId": self.model_id,
            "messages": [{"role": "user", "content": [{"text": prompt}]}],
            "outputConfig": {"textFormat": {"type": "json_schema", "structure": {"jsonSchema": {
                "name": schema_model.__name__, "description": schema_model.__doc__ or "", "schema": json.dumps(schema),
            }}}},
        }
        if system:
            kwargs["system"] = [{"text": system}]
        resp = self.client.converse(**kwargs)
        raw = resp["output"]["message"]["content"][0]["text"]
        return schema_model.model_validate_json(raw)

Los demás proveedores quedan como ejercicio — solo necesitan implementar `generate()` (ya lo tenés resuelto de la Clase 2, sección "Tu proveedor"); `generate_structured()` viene gratis de la clase base.

In [ ]:
class AnthropicProvider(LLMProvider):
    def __init__(self):
        import anthropic
        self.client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
        self.model_id = os.environ.get("ANTHROPIC_MODEL_ID", "claude-sonnet-4-6")

    def generate(self, prompt, system=None, temperature=0.7, max_tokens=500):
        # TODO: reutilizar la lógica de llamar_anthropic (Clase 2, sección 2)
        raise NotImplementedError


class OpenAIProvider(LLMProvider):
    def __init__(self):
        from openai import OpenAI
        self.client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
        self.model_id = os.environ.get("OPENAI_MODEL_ID", "gpt-4o")

    def generate(self, prompt, system=None, temperature=0.7, max_tokens=500):
        # TODO: reutilizar la lógica de llamar_openai (Clase 2, sección 2)
        raise NotImplementedError


class GoogleProvider(LLMProvider):
    def __init__(self):
        from google import genai
        self.client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
        self.model_id = os.environ.get("GOOGLE_MODEL_ID", "gemini-2.0-flash")

    def generate(self, prompt, system=None, temperature=0.7, max_tokens=500):
        # TODO: reutilizar la lógica de llamar_google (Clase 2, sección 2)
        raise NotImplementedError


class OllamaProvider(LLMProvider):
    def __init__(self):
        self.model_id = os.environ.get("OLLAMA_MODEL_ID", "llama3.1")

    def generate(self, prompt, system=None, temperature=0.7, max_tokens=500):
        # TODO: reutilizar la lógica de llamar_ollama (Clase 2, sección 2)
        raise NotImplementedError


def get_provider() -> LLMProvider:
    """Factory: devuelve la instancia del proveedor configurado en LLM_PROVIDER."""
    proveedores = {
        "bedrock": BedrockProvider,
        "anthropic": AnthropicProvider,
        "openai": OpenAIProvider,
        "google": GoogleProvider,
        "ollama": OllamaProvider,
    }
    nombre = os.environ.get("LLM_PROVIDER", "bedrock")
    if nombre not in proveedores:
        raise ValueError(f"Proveedor desconocido: {nombre}")
    return proveedores[nombre]()


# Prueba: esto funciona hoy con Bedrock. Cuando completes los TODO de arriba, va a funcionar
# igual (sin cambiar esta celda) con cualquier otro proveedor que configures en LLM_PROVIDER.
llm = get_provider()
print(llm.generate("Decí 'funciona' y nada más."))

**🏠 Practicá**: completá al menos un `TODO` (el de tu proveedor real, si no es Bedrock) y confirmá que `get_provider().generate(...)` funciona sin tocar ninguna otra celda del notebook. Después probá `get_provider().generate_structured("Clasificá esta consulta: 'no me llegó la factura'", ConsultaSoporte)` — con Bedrock usa el mecanismo nativo (sección 2); con cualquier otro proveedor, usa el reintento genérico (sección 3) automáticamente, sin que hayas escrito nada extra para eso.

## 9. Laboratorio: servicio de extracción estructurada

**🏠 Tarea de la semana — entregar/completar antes de la Clase 4.** El docente presenta la consigna y muestra el patrón en vivo, pero la resolución es trabajo individual asincrónico.

**Checkpoint del proyecto integrador (Clase 3)**: el `LLMClient` de la Clase 2, ahora detrás de la interfaz `LLMProvider`, más un servicio que transforma texto libre en objetos validados.

### Consigna

1. Terminá de completar los `TODO` de `AnthropicProvider` / `OpenAIProvider` / `GoogleProvider` / `OllamaProvider` — al menos el de **tu** proveedor.
2. Definí un modelo Pydantic propio (no `ConsultaSoporte`) relevante para el dominio del TP integrador — por ejemplo, una clasificación de la intención de una pregunta sobre la documentación técnica del corpus (`tipo_pregunta`, `requiere_busqueda_en_documentacion: bool`, `resumen`).
3. Escribí una función `clasificar(texto: str) -> TuModelo` que use `get_provider().generate_structured(...)` para resolverlo.
4. Probala con al menos 3 textos de entrada distintos, incluyendo uno ambiguo o mal escrito, y confirmá que **siempre** devuelve un objeto válido o levanta una excepción clara (nunca un `dict` sin validar ni un texto crudo).

### Criterio de éxito

- El servicio nunca deja pasar un objeto que no cumple tu schema Pydantic — o devuelve el objeto validado, o falla explícitamente.
- Funciona con el proveedor configurado en tu `.env`, sin credenciales hardcodeadas.
- Podés explicar en una frase la diferencia entre usar `generate_structured` con Bedrock (nativo) y con cualquier otro proveedor (reintento + prompt) — sin que el código que llama a `clasificar()` necesite saberlo.

In [ ]:
class MiModelo(BaseModel):
    # TODO: definí los campos relevantes para tu caso de uso del TP
    pass


def clasificar(texto: str) -> "MiModelo":
    # TODO: usar get_provider().generate_structured(...)
    raise NotImplementedError


# Prueba (va a fallar hasta que completes los TODOs):
# print(clasificar("¿Cómo configuro las credenciales de Bedrock?"))

## Cierre

- Hoy el `LLMClient` de la Clase 2 se convirtió en algo bastante más sólido: interfaz común (`LLMProvider`), salidas estructuradas y validadas, tool calling, timeouts, reintentos con jitter, registro de costos y streaming.
- Esto **es** el primer entregable serio del TP integrador — de acá en adelante (RAG, agentes, despliegue) todo se construye sobre esta base, no se vuelve a escribir.
- Antes de la próxima clase: dejar el laboratorio de la sección 9 funcionando con tu proveedor.
- Próxima clase: embeddings e ingesta documental — empezamos a preparar el corpus para la recuperación semántica.